# Proyecto 1 — Dashboard de Ventas y Logística
## Punto 9 — Análisis SQL y KPIs
**Autor:** Michael Hans Evans  
**Input:** `superstore_model.db` (13 tablas, Snowflake 3FN)  
**Herramienta principal:** SQL / SQLite  
**Validación:** Python / pandas (validación cruzada)

---
> Este notebook ejecuta el Punto 9 del proyecto — Análisis SQL.  
> Cada sección responde una hipótesis planteada en el Punto 3.  
> Los resultados son validados contra Python/pandas y el dashboard Power BI.  
> Buenas prácticas: CTEs, aliases descriptivos, comentarios por sección.

## 0. Setup — Conexión y librerías

In [1]:
import sqlite3
import pandas as pd

# mode=ro evita conflictos de lock con OneDrive
conn = sqlite3.connect('file:../database/superstore_model.db?mode=ro', uri=True)
cur = conn.cursor()

tables = [t[0] for t in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
print(f"Tablas en superstore_model.db: {len(tables)}")
for t in tables:
    n = cur.execute(f"SELECT COUNT(*) FROM '{t}'").fetchone()[0]
    print(f"  {t}: {n:,}")

Tablas en superstore_model.db: 13
  dim_categories: 3
  dim_customers: 793
  dim_customers_segment: 3
  dim_geography: 632
  dim_geography_city: 531
  dim_geography_region: 4
  dim_geography_state: 49
  dim_products: 1,862
  dim_products_detail: 1,862
  dim_shipment: 4
  dim_sub_categories: 17
  dim_time: 5,009
  fact_orders: 9,993


## 9.1 Validación del modelo — KPIs generales

In [2]:
query = '''
SELECT
    ROUND(SUM(Sales), 2)                            AS total_sales,
    ROUND(SUM(Profit), 2)                           AS total_profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2)        AS profit_margin_pct,
    COUNT(*)                                        AS total_registros,
    ROUND(AVG(Sales), 2)                            AS avg_ticket,
    ROUND(AVG(Discount) * 100, 2)                   AS avg_discount_pct,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END)     AS ordenes_con_perdida
FROM fact_orders
'''
pd.read_sql(query, conn).T

,0
total_sales,2296919.49
total_profit,286409.08
profit_margin_pct,12.47
total_registros,9993.00
avg_ticket,229.85
avg_discount_pct,15.62
ordenes_con_perdida,1870.00


## 9.2 Análisis por Hipótesis

### H1 — Rentabilidad por categoría
> *Suponemos que no todas las categorías generan rentabilidad positiva y que existe al menos una subcategoría con pérdidas sistemáticas a pesar de tener volumen considerable.*

In [3]:
query_h1_1 = '''
SELECT
    dc.Category,
    ROUND(SUM(fo.Sales), 2)                         AS total_sales,
    ROUND(SUM(fo.Profit), 2)                        AS total_profit,
    ROUND(SUM(fo.Profit) / SUM(fo.Sales) * 100, 2) AS profit_margin_pct,
    SUM(CASE WHEN fo.Profit < 0 THEN 1 ELSE 0 END) AS ordenes_con_perdida
FROM fact_orders fo
JOIN dim_products_detail dpd ON fo."Product ID" = dpd."Product ID"
JOIN dim_categories dc       ON dpd.categories_key = dc.categories_key
GROUP BY dc.Category
ORDER BY profit_margin_pct DESC
'''
pd.read_sql(query_h1_1, conn)

,Category,total_sales,total_profit,profit_margin_pct,ordenes_con_perdida
0,Technology,836154.03,145454.95,17.40,271
1,Office Supplies,719047.03,122490.80,17.04,886
2,Furniture,741718.42,18463.33,2.49,713


In [4]:
query_h1_2 = '''
SELECT
    dsc."Sub-Category",
    dc.Category,
    ROUND(SUM(fo.Sales), 2)                         AS total_sales,
    ROUND(SUM(fo.Profit), 2)                        AS total_profit,
    ROUND(SUM(fo.Profit) / SUM(fo.Sales) * 100, 2) AS profit_margin_pct,
    ROUND(SUM(CASE WHEN fo.Profit < 0 THEN 1.0 ELSE 0 END)/COUNT(*)*100, 2) AS pct_ordenes_perdida
FROM fact_orders fo
JOIN dim_products_detail dpd ON fo."Product ID" = dpd."Product ID"
JOIN dim_categories dc       ON dpd.categories_key = dc.categories_key
JOIN dim_sub_categories dsc  ON dpd.sub_categories_key = dsc.sub_categories_key
GROUP BY dsc."Sub-Category", dc.Category
ORDER BY profit_margin_pct ASC
LIMIT 5
'''
pd.read_sql(query_h1_2, conn)

,Sub-Category,Category,total_sales,total_profit,profit_margin_pct,pct_ordenes_perdida
0,Tables,Furniture,206965.53,-17725.48,-8.56,63.64
1,Bookcases,Furniture,114880.00,-3472.56,-3.02,47.81
2,Supplies,Office Supplies,46673.54,-1189.10,-2.55,17.37
3,Machines,Technology,189238.63,3384.76,1.79,38.26
4,Chairs,Furniture,328167.73,26602.23,8.11,37.99


**H1 confirmada:** Furniture es la categoría con peor margen (2.49%). Tables (-8.56%) y Bookcases (-3.02%) presentan pérdidas sistemáticas — el 63.64% de las órdenes de Tables generan pérdida.

### H2 — Impacto del descuento en la rentabilidad
> *Suponemos que existe una relación negativa entre el nivel de descuento aplicado y el profit generado, y que a partir de un cierto umbral de descuento las ventas generan pérdidas.*

In [5]:
query_h2 = '''
SELECT
    CASE
        WHEN Discount = 0     THEN '1. Sin descuento (0%)'
        WHEN Discount <= 0.20 THEN '2. Bajo (1-20%)'
        WHEN Discount <= 0.40 THEN '3. Medio (21-40%)'
        WHEN Discount <= 0.60 THEN '4. Alto (41-60%)'
        ELSE                       '5. Muy alto (61-80%)'
    END AS rango_descuento,
    COUNT(*)                                                AS total_ordenes,
    ROUND(AVG(Profit), 2)                                   AS avg_profit,
    ROUND(SUM(Profit)/SUM(Sales)*100, 2)                    AS margin_pct,
    ROUND(SUM(CASE WHEN Profit < 0 THEN 1.0 ELSE 0 END)/COUNT(*)*100, 2) AS pct_perdida
FROM fact_orders
GROUP BY rango_descuento
ORDER BY rango_descuento
'''
pd.read_sql(query_h2, conn)

,rango_descuento,total_ordenes,avg_profit,margin_pct,pct_perdida
0,1. Sin descuento (0%),4798,66.90,29.51,0.00
1,2. Bajo (1-20%),3803,26.50,11.91,13.75
2,3. Medio (21-40%),459,-78.01,-15.31,90.20
3,4. Alto (41-60%),215,-134.62,-40.74,100.00
4,5. Muy alto (61-80%),718,-98.35,-122.63,100.00


**H2 confirmada:** A partir del 21% de descuento el margen se vuelve negativo (-15.31%). En el rango 41-80%, el 100% de las órdenes generan pérdida.

### H3 — Performance regional
> *Suponemos que existe al menos una región que genera alto volumen de ventas pero baja rentabilidad, indicando ineficiencias comerciales o logísticas en esa zona.*

In [6]:
query_h3 = '''
SELECT
    dgr.Region,
    ROUND(SUM(fo.Sales), 2)                         AS total_sales,
    ROUND(SUM(fo.Profit), 2)                        AS total_profit,
    ROUND(SUM(fo.Profit) / SUM(fo.Sales) * 100, 2) AS profit_margin_pct,
    COUNT(*)                                        AS total_ordenes
FROM fact_orders fo
JOIN dim_geography dg         ON fo.geography_key = dg.geography_key
JOIN dim_geography_region dgr ON dg.region_key = dgr.region_key
GROUP BY dgr.Region
ORDER BY profit_margin_pct DESC
'''
pd.read_sql(query_h3, conn)

,Region,total_sales,total_profit,profit_margin_pct,total_ordenes
0,West,725457.82,108418.45,14.94,3203
1,East,678499.87,91534.84,13.49,2847
2,South,391721.91,46749.43,11.93,1620
3,Central,501239.89,39706.36,7.92,2323


**H3 confirmada:** Central es 3ra en ventas pero tiene el peor margen (7.92%), casi la mitad de West (14.94%). Indica ineficiencias comerciales o logísticas en esa región.

### H4 — Logística y modo de envío
> *Suponemos que Standard Class concentra la mayor parte de los pedidos y que Corporate es el segmento que más utiliza envíos express.*

In [7]:
query_h4_1 = '''
SELECT
    ds."Ship Mode",
    COUNT(*) AS total_ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_ordenes,
    ROUND(AVG(fo.Shipping_Days), 1) AS avg_dias_envio
FROM fact_orders fo
JOIN dim_shipment ds ON fo.shipment_key = ds.shipment_key
GROUP BY ds."Ship Mode"
ORDER BY total_ordenes DESC
'''
pd.read_sql(query_h4_1, conn)

,Ship Mode,total_ordenes,pct_ordenes,avg_dias_envio
0,Standard Class,5967,59.71,5.0
1,Second Class,1945,19.46,3.2
2,First Class,1538,15.39,2.2
3,Same Day,543,5.43,0.0


In [8]:
query_h4_2 = '''
SELECT
    dcs.Segment,
    ds."Ship Mode",
    COUNT(*) AS total_ordenes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY dcs.Segment), 2) AS pct_en_segmento
FROM fact_orders fo
JOIN dim_customers dc          ON fo."Customer ID" = dc."Customer ID"
JOIN dim_customers_segment dcs ON dc.segment_key = dcs.segment_key
JOIN dim_shipment ds           ON fo.shipment_key = ds.shipment_key
GROUP BY dcs.Segment, ds."Ship Mode"
ORDER BY dcs.Segment, total_ordenes DESC
'''
pd.read_sql(query_h4_2, conn)

,Segment,Ship Mode,total_ordenes,pct_en_segmento
0,Consumer,Standard Class,3085,59.43
1,Consumer,Second Class,1020,19.65
2,Consumer,First Class,769,14.81
3,Consumer,Same Day,317,6.11
4,Corporate,Standard Class,1812,60.00
5,Corporate,Second Class,609,20.17
6,Corporate,First Class,485,16.06
7,Corporate,Same Day,114,3.77
8,Home Office,Standard Class,1070,60.04
9,Home Office,Second Class,316,17.73


**H4 parcialmente confirmada:** Standard Class domina con 59.71% del total. La distribución por segmento es homogénea (~60% Standard en todos). Corporate NO lidera en envíos express — Same Day es más frecuente en Consumer (6.11%) y Home Office (6.29%) que en Corporate (3.77%).

### H5 — Clientes más valiosos
> *Suponemos que el 20% de los clientes genera el 80% de los ingresos y que existen clientes con alto volumen pero márgenes muy bajos.*

In [9]:
query_h5_1 = '''
SELECT
    dc."Customer Name",
    dcs.Segment,
    ROUND(SUM(fo.Sales), 2)                         AS total_sales,
    ROUND(SUM(fo.Profit), 2)                        AS total_profit,
    ROUND(SUM(fo.Profit) / SUM(fo.Sales) * 100, 2) AS profit_margin_pct
FROM fact_orders fo
JOIN dim_customers dc          ON fo."Customer ID" = dc."Customer ID"
JOIN dim_customers_segment dcs ON dc.segment_key = dcs.segment_key
GROUP BY dc."Customer ID", dc."Customer Name", dcs.Segment
ORDER BY total_sales DESC
LIMIT 10
'''
pd.read_sql(query_h5_1, conn)

,Customer Name,Segment,total_sales,total_profit,profit_margin_pct
0,Sean Miller,Home Office,25043.05,-1980.74,-7.91
1,Tamara Chand,Corporate,19052.22,8981.32,47.14
2,Raymond Buch,Consumer,15117.34,6976.10,46.15
3,Tom Ashbrook,Home Office,14595.62,4703.79,32.23
4,Adrian Barton,Consumer,14473.57,5444.81,37.62
5,Ken Lonsdale,Consumer,14175.23,806.85,5.69
6,Sanjit Chand,Consumer,14142.33,5757.41,40.71
7,Hunter Lopez,Consumer,12873.30,5622.43,43.68
8,Sanjit Engle,Consumer,12209.44,2650.68,21.71
9,Christopher Conant,Consumer,12129.07,2177.05,17.95


In [10]:
query_h5_2 = '''
SELECT
    dc."Customer Name",
    dcs.Segment,
    ROUND(SUM(fo.Sales), 2)  AS total_sales,
    ROUND(SUM(fo.Profit), 2) AS total_profit
FROM fact_orders fo
JOIN dim_customers dc          ON fo."Customer ID" = dc."Customer ID"
JOIN dim_customers_segment dcs ON dc.segment_key = dcs.segment_key
GROUP BY dc."Customer ID", dc."Customer Name", dcs.Segment
HAVING SUM(fo.Profit) < 0
ORDER BY total_profit ASC
LIMIT 5
'''
pd.read_sql(query_h5_2, conn)

,Customer Name,Segment,total_sales,total_profit
0,Cindy Stewart,Consumer,5690.05,-6626.39
1,Grant Thornton,Corporate,9351.21,-4108.66
2,Luke Foster,Consumer,3930.51,-3583.98
3,Sharelle Roach,Home Office,3233.48,-3333.91
4,Henry Goldwyn,Corporate,3247.64,-2797.96


**H5 confirmada:** Sean Miller es el mayor comprador ($25,043.05) pero genera pérdida neta (-$1,980.74, margen -7.91%). Cindy Stewart es la peor pérdida absoluta (-$6,626.39). Existen clientes de alto volumen con márgenes negativos.

### H6 — Estacionalidad de las ventas
> *Suponemos que las ventas no se distribuyen uniformemente a lo largo del año y que existe un patrón estacional consistente que impacta en la planificación del negocio.*

In [11]:
query_h6_1 = '''
SELECT
    dt.Order_Quarter,
    ROUND(SUM(fo.Sales), 2) AS total_sales,
    ROUND(SUM(fo.Sales) * 100.0 / SUM(SUM(fo.Sales)) OVER (), 2) AS pct_anual
FROM fact_orders fo
JOIN dim_time dt ON fo."Order ID" = dt."Order ID"
GROUP BY dt.Order_Quarter
ORDER BY dt.Order_Quarter
'''
pd.read_sql(query_h6_1, conn)

,Order_Quarter,total_sales,pct_anual
0,1,359681.58,15.66
1,2,445228.25,19.38
2,3,613932.11,26.73
3,4,878077.56,38.23


In [12]:
query_h6_2 = '''
SELECT
    dt.Order_Year,
    dt.Order_Quarter,
    ROUND(SUM(fo.Sales), 2)  AS total_sales,
    ROUND(SUM(fo.Profit), 2) AS total_profit
FROM fact_orders fo
JOIN dim_time dt ON fo."Order ID" = dt."Order ID"
GROUP BY dt.Order_Year, dt.Order_Quarter
ORDER BY dt.Order_Year, dt.Order_Quarter
'''
pd.read_sql(query_h6_2, conn)

,Order_Year,Order_Quarter,total_sales,total_profit
0,2014,1,74447.80,3811.23
1,2014,2,86257.39,11216.13
2,2014,3,143633.21,12804.72
3,2014,4,179627.73,21723.95
4,2015,1,68851.74,9264.94
5,2015,2,89124.19,12190.92
6,2015,3,130259.58,16853.62
7,2015,4,182297.01,23309.12
8,2016,1,93237.18,11441.37
9,2016,2,136082.30,16390.34


**H6 confirmada:** Q4 concentra el 38.23% de las ventas anuales. Patrón consistente 2014–2017 con crecimiento progresivo Q1→Q4.

## 9.3 Validación cruzada — SQL vs Python

In [13]:
DATA_URL = (
    "https://raw.githubusercontent.com/MHEvans02/"
    "Dashboard_Ventas_Logistica/main/data/superstore_clean.csv"
)
df = pd.read_csv(DATA_URL)

checks = {
    "Total Sales":    (cur.execute("SELECT ROUND(SUM(Sales),2) FROM fact_orders").fetchone()[0],
                       round(df['Sales'].sum(), 2), 2296919.49),
    "Total Profit":   (cur.execute("SELECT ROUND(SUM(Profit),2) FROM fact_orders").fetchone()[0],
                       round(df['Profit'].sum(), 2), 286409.08),
    "Customers":      (cur.execute("SELECT COUNT(*) FROM dim_customers").fetchone()[0],
                       df['Customer ID'].nunique(), 793),
    "Registros":      (cur.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0],
                       len(df), 9993),
    "Órd. Pérdida":   (cur.execute("SELECT COUNT(*) FROM fact_orders WHERE Profit < 0").fetchone()[0],
                       (df['Profit'] < 0).sum(), 1870),
}

resultados = []
for kpi, (sql_v, py_v, pbi_v) in checks.items():
    ok = abs(float(sql_v)-float(pbi_v)) < 0.05 and abs(float(py_v)-float(pbi_v)) < 0.05
    resultados.append({'KPI': kpi, 'SQL': sql_v, 'Python': py_v, 'Power BI': pbi_v, 'OK': ok})

conn.close()

df_val = pd.DataFrame(resultados).set_index('KPI')
print(f"{'Validación cruzada OK' if df_val['OK'].all() else 'HAY DISCREPANCIAS'}")
df_val

Validación cruzada OK


,SQL,Python,Power BI,OK
KPI,,,,
Total Sales,2296919.49,2296919.49,2296919.49,True
Total Profit,286409.08,286409.08,286409.08,True
Customers,793.00,793.00,793.00,True
Registros,9993.00,9993.00,9993.00,True
Órd. Pérdida,1870.00,1870.00,1870.00,True
